# Linear Regression

**Topic:** Supervised Learning — Regression

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, FloatSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how linear regression finds the best-fit line by minimizing squared errors
- **Explain** what the slope and intercept coefficients represent in plain English
- **Interpret** a residual plot to diagnose whether the linear assumption holds

> **Tip:** In the Math Explorer widget below, drag the slope and intercept sliders and watch how the MSE value responds. The minimum MSE is exactly what the normal equations solve for in one step.

---
## How we got here

Linear regression is the foundation of nearly every algorithm in this folder. Before working through it, make sure you are comfortable with:

- **[math/calculus/08_the_cost_function.ipynb](../math/calculus/08_the_cost_function.ipynb)** — what a loss function is and why MSE is a natural choice for regression
- **[math/linear_algebra/09_linear_algebra_in_ml.ipynb](../math/linear_algebra/09_linear_algebra_in_ml.ipynb)** — how matrices represent many training examples at once
- **[math/statistics_for_ml/03_the_normal_equations.ipynb](../math/statistics_for_ml/03_the_normal_equations.ipynb)** — the closed-form solution that solves regression exactly

Those three notebooks give you the mathematical vocabulary to read the equations below without confusion.

---
## Why this matters for data science

Linear regression is not just a simple starting point. It is often the right endpoint. When your target variable has a roughly linear relationship with your features, a well-tuned linear model is fast to train, easy to interpret, and hard to beat in production.

More importantly, every more complex algorithm in this folder is built on ideas that linear regression introduces: a loss function, a learning objective, a set of parameters to optimize, and the bias-variance tradeoff. Master this one and the rest become variations on a theme.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Linear regression sits at the **far top-left**: maximum interpretability, minimum complexity. Every coefficient has a direct, plain-English meaning: "a one-unit increase in feature X is associated with a β-unit change in y, holding all other features constant." There is no black box.

The trade-off is that it assumes a linear relationship between features and target. When that assumption breaks down — because the true relationship is curved or the features interact in complex ways — the model underfits and a more flexible algorithm may be needed.

---
## How it learns

Imagine you have a scatter plot of points: median income on the x-axis, house value on the y-axis. Your job is to draw the single straight line that comes closest to all the points at once.

"Closest" means minimizing the total squared vertical distance between each point and the line. Those vertical distances are called **residuals**: the gap between what the model predicted and what actually happened.

The algorithm tries different slopes and intercepts until it finds the combination that makes the total squared residuals as small as possible. That minimum exists at exactly one location, and the normal equations find it directly without any trial and error.

Once the line is drawn, the slope tells you how much y changes for each one-unit increase in x. The intercept tells you the predicted y when x is zero (which is only meaningful if zero is a realistic input value).

---
## The math behind it

Linear regression minimizes **mean squared error (MSE)**:

$$J(\boldsymbol{\theta}) = \frac{1}{n} \sum_{i=1}^{n} \left(\hat{y}_i - y_i\right)^2$$

- $n$ — number of training examples
- $\hat{y}_i = \theta_0 + \theta_1 x_{i1} + \cdots + \theta_p x_{ip}$ — predicted value for example $i$
- $y_i$ — true target value for example $i$
- $\boldsymbol{\theta} = [\theta_0, \theta_1, \ldots, \theta_p]$ — vector of coefficients to learn

**Closed-form solution (Normal Equations)** — finds the exact minimum of MSE in one step:

$$\boldsymbol{\theta} = (X^\top X)^{-1} X^\top \mathbf{y}$$

- $X$ — design matrix of shape $(n \times p{+}1)$; first column is all ones for the intercept
- $\mathbf{y}$ — vector of target values, shape $(n \times 1)$

**Prediction on new data:**

$$\hat{y} = \boldsymbol{\theta}^\top \mathbf{x} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p$$

The optimization criterion is minimizing MSE. Sklearn uses the closed-form solution for small datasets and a numerically stable variant (SVD decomposition) for large or ill-conditioned ones.

---
## Assumptions of Linear Regression

Linear regression makes five assumptions about your data. Violating them does not always break the model, but it does change what you can trust about the output.

### 1. Linearity

The relationship between each feature and the target follows a straight line.

- **How to check:** Residuals vs Fitted plot — random scatter around zero means the assumption holds
- **Violation looks like:** A curved pattern in the residuals (U-shape or arch)
- **Consequence:** Coefficients are biased and predictions are systematically wrong in some regions

### 2. Independence

Each observation is independent of all others; knowing one tells you nothing about another.

- **How to check:** Durbin-Watson test — a value near 2.0 indicates no autocorrelation
- **Common violation:** Time series data, where consecutive observations are correlated by definition
- **Consequence:** Standard errors are underestimated and p-values become unreliable

### 3. Homoscedasticity

The spread (variance) of the residuals is consistent across all fitted values.

- **How to check:** Scale-Location plot — a flat horizontal band indicates constant variance
- **Violation looks like:** A funnel shape (residuals fan out as fitted values increase)
- **Consequence:** Coefficient estimates are still unbiased but standard errors are wrong, making confidence intervals and p-values unreliable

### 4. Normality of Residuals

The residuals — not the raw data — follow a normal distribution.

- **How to check:** Q-Q plot of residuals, or a Shapiro-Wilk test
- **Important:** This assumption is about residuals, not about X or y directly. Checking normality of the raw features or target is a common misconception
- **Consequence:** With small samples, confidence intervals and hypothesis tests rely on this; with large samples the Central Limit Theorem makes the model robust to violations

### 5. No Multicollinearity

Features are not highly correlated with each other.

- **How to check:** Correlation matrix, Variance Inflation Factor (VIF)
- **Violation looks like:** Unstable coefficients with large standard errors — adding or removing one feature drastically shifts another's coefficient
- **Consequence:** Individual coefficients become unreliable and hard to interpret, even when the model's predictions are still accurate

---
## Residual Analysis in Depth

A residual is the difference between the actual value and the model's prediction:

$$e_i = y_i - \hat{y}_i$$

**Standardized residuals** divide each residual by the estimated standard deviation of all residuals, making them comparable across different scales. Any standardized residual with absolute value above 3 is worth investigating as a potential outlier.

### The three diagnostic plots

| Plot | Checks | Good pattern | Warning pattern |
|------|--------|-------------|----------------|
| Residuals vs Fitted | Linearity and homoscedasticity | Random scatter around y = 0 | Curved trend or funnel shape |
| Q-Q Plot | Normality of residuals | Points fall on the diagonal | S-curve or heavy tails |
| Scale-Location | Homoscedasticity | Flat horizontal band | Upward slope (variance increases) |

The three charts below use a 2,000-observation sample from California Housing.

In [2]:
import statsmodels.api as sm
from scipy import stats as sp_stats
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
_h = fetch_california_housing(as_frame=True)
_X, _y = _h.data.values, _h.target.values
_idx = np.random.choice(len(_X), 2000, replace=False)
_Xd, _yd = _X[_idx], _y[_idx]

_Xc      = sm.add_constant(_Xd)
_sm      = sm.OLS(_yd, _Xc).fit()
_inf     = _sm.get_influence()
_fitted  = _sm.fittedvalues
_resid   = _sm.resid
_sresid  = _inf.resid_studentized_internal

# 1. Residuals vs Fitted
_f1 = go.Figure(
    data=go.Scatter(x=_fitted, y=_resid, mode="markers",
        marker=dict(color=PALETTE["primary"], size=4, opacity=0.5), name="Residual"),
    layout=base_layout("1. Residuals vs Fitted — checks linearity and homoscedasticity",
        "Fitted values", "Residuals"),
)
_f1.add_hline(y=0, line_color=PALETTE["muted"], line_width=1.5, line_dash="dash")
_f1.show()

# 2. Q-Q Plot
(osm, osr), _ = sp_stats.probplot(_resid)
_qs, _qi = np.polyfit(osm, osr, 1)
_qx = np.array([float(osm.min()), float(osm.max())])
_f2 = go.Figure(layout=base_layout("2. Q-Q Plot — checks normality of residuals",
    "Theoretical quantiles", "Sample quantiles"))
_f2.add_trace(go.Scatter(x=osm, y=osr, mode="markers",
    marker=dict(color=PALETTE["primary"], size=3, opacity=0.5), name="Residuals"))
_f2.add_trace(go.Scatter(x=_qx, y=_qs * _qx + _qi, mode="lines",
    line=dict(color=PALETTE["muted"], width=2, dash="dash"), name="Reference line"))
_f2.show()

# 3. Scale-Location
_f3 = go.Figure(
    data=go.Scatter(x=_fitted, y=np.sqrt(np.abs(_sresid)), mode="markers",
        marker=dict(color=PALETTE["primary"], size=4, opacity=0.5),
        name="sqrt|Std residual|"),
    layout=base_layout("3. Scale-Location — checks homoscedasticity",
        "Fitted values", "sqrt|Standardized residuals|"),
)
_f3.add_hline(y=float(np.sqrt(np.abs(_sresid)).mean()),
              line_color=PALETTE["secondary"], line_width=2, line_dash="dash",
              annotation_text="mean level")
_f3.show()

---
## Confidence Intervals vs Prediction Intervals

When you ask "what will this model predict for x = 5?", there are two different uncertainty questions you might be asking.

**Confidence interval:** How uncertain are we about the *average* response at this x value? This captures uncertainty about where the true regression line sits.

> "What is the average house value for neighborhoods with median income of 8?"

**Prediction interval:** How uncertain are we about a *single new observation* at this x value? This captures uncertainty about both the regression line and the natural scatter of individual points around it.

> "What will this specific neighborhood with median income 8 sell for?"

Prediction intervals are always wider because they account for two sources of uncertainty: where the line is (same as CI) plus the irreducible variability of individual data points around the line.

$$\text{CI: } \hat{y} \pm t^* \cdot s\sqrt{\frac{1}{n} + \frac{(x^* - \bar{x})^2}{S_{xx}}}$$

$$\text{PI: } \hat{y} \pm t^* \cdot s\sqrt{1 + \frac{1}{n} + \frac{(x^* - \bar{x})^2}{S_{xx}}}$$

The only difference is the extra 1 under the square root in the PI. Even with infinite data the PI never collapses to a point — but the CI does.

See **[statistics/16_confidence_intervals.ipynb](../statistics/16_confidence_intervals.ipynb)** for the conceptual foundation of confidence intervals.

| Interval type | What it captures | Width | When to use it |
|---|---|---|---|
| Confidence interval | Uncertainty about the mean response | Narrower — only parameter uncertainty | Estimating average outcome for a group |
| Prediction interval | Uncertainty about a single new observation | Wider — parameter uncertainty plus natural variation | Forecasting an individual outcome |

In [3]:
from scipy import stats as sp_stats2
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

np.random.seed(42)
_hc = fetch_california_housing(as_frame=True)
_xc = _hc.data["MedInc"].values
_yc = _hc.target.values

_lr_1d = LinearRegression().fit(_xc.reshape(-1, 1), _yc)
_sl, _ic = float(_lr_1d.coef_[0]), float(_lr_1d.intercept_)

_nc    = len(_xc)
_xmean = _xc.mean()
_Sxx   = float(np.sum((_xc - _xmean) ** 2))
_yhat  = _lr_1d.predict(_xc.reshape(-1, 1))
_s2    = float(np.sum((_yc - _yhat) ** 2) / (_nc - 2))
_tcrit = sp_stats2.t.ppf(0.975, df=_nc - 2)

_xnew   = np.linspace(_xc.min(), _xc.max(), 300)
_ynew   = _ic + _sl * _xnew
_se_ci  = np.sqrt(_s2 * (1 / _nc + (_xnew - _xmean) ** 2 / _Sxx))
_se_pi  = np.sqrt(_s2 * (1 + 1 / _nc + (_xnew - _xmean) ** 2 / _Sxx))

_ci_lo, _ci_hi = _ynew - _tcrit * _se_ci, _ynew + _tcrit * _se_ci
_pi_lo, _pi_hi = _ynew - _tcrit * _se_pi, _ynew + _tcrit * _se_pi

_idxs = np.random.choice(_nc, 600, replace=False)

_fci = go.Figure()
_fci.add_trace(go.Scatter(x=_xc[_idxs], y=_yc[_idxs], mode="markers",
    marker=dict(color=PALETTE["muted"], size=4, opacity=0.3), name="Data (sample)"))
_fci.add_trace(go.Scatter(
    x=np.concatenate([_xnew, _xnew[::-1]]),
    y=np.concatenate([_pi_hi, _pi_lo[::-1]]),
    fill="toself", fillcolor="rgba(247,103,7,0.10)",
    line=dict(color=PALETTE["secondary"], width=0), name="95% Prediction Interval"))
_fci.add_trace(go.Scatter(
    x=np.concatenate([_xnew, _xnew[::-1]]),
    y=np.concatenate([_ci_hi, _ci_lo[::-1]]),
    fill="toself", fillcolor="rgba(76,110,245,0.20)",
    line=dict(color=PALETTE["primary"], width=0), name="95% Confidence Interval"))
_fci.add_trace(go.Scatter(x=_xnew, y=_ynew, mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5), name="Regression line"))
_fci.update_layout(base_layout(
    "Confidence vs Prediction Interval — MedInc predicting house value",
    "Median Income (tens of thousands $)",
    "Median House Value (hundreds of thousands $)"))
_fci.show()

---
## Reading OLS Output

The statsmodels OLS summary is one of the most information-dense outputs in statistical modeling. Here is how to produce it and what every number means.

```python
import statsmodels.api as sm
X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()
print(model.summary())
```

### Header section

**R-squared:** The proportion of variance in y explained by the model. R² = 0.60 means the model explains 60% of the variation in house prices; the remaining 40% is unexplained.

**Adj. R-squared:** R² penalized for the number of predictors. Adding any feature — even a random one — always increases R². Adjusted R² only increases if the new feature explains more than expected by chance. Use this for comparing models with different numbers of features.

**F-statistic:** Tests whether the model as a whole is statistically significant. It asks: is at least one coefficient nonzero, compared to a baseline model with only an intercept?

**Prob(F-statistic):** The p-value for the F-test. Values below 0.05 mean the model as a whole is significant.

### Coefficients table

**coef:** The slope for each feature. Interpretation: "Holding all other features constant, a one-unit increase in this feature is associated with a coef-unit change in y."

**std err:** The standard error of the coefficient estimate — the uncertainty around the coefficient. Large std err relative to coef means the estimate is imprecise.

**t:** The coefficient divided by its standard error. A large absolute t-value means the coefficient is many standard errors away from zero.

**P>|t|:** The p-value for each coefficient. Tests H\u2080: this coefficient = 0.
- p < 0.05: statistically significant
- p > 0.05: cannot reject H\u2080 — the data does not provide strong evidence of an effect

**[0.025, 0.975]:** The 95% confidence interval for the coefficient. If this range includes zero, the coefficient is not significantly different from zero at the 5% level.

### Diagnostics

**Durbin-Watson:** Tests for autocorrelation in the residuals. A value near 2.0 indicates no autocorrelation.

**Omnibus / Jarque-Bera:** Tests whether the residuals are normally distributed. A significant p-value (< 0.05) suggests non-normality.

**Condition Number:** Tests for multicollinearity. Values above 30 suggest potential issues; above 1000 indicates serious multicollinearity.

In [4]:
import statsmodels.api as sm
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
_ho = fetch_california_housing(as_frame=True)
_Xo, _yo = _ho.data, _ho.target
_Xoc = sm.add_constant(_Xo)
_ols = sm.OLS(_yo, _Xoc).fit()
print(_ols.summary())

                            OLS Regression Results                            
Dep. Variable:            MedHouseVal   R-squared:                       0.606
Model:                            OLS   Adj. R-squared:                  0.606
Method:                 Least Squares   F-statistic:                     3970.
Date:                Tue, 30 Jun 2026   Prob (F-statistic):               0.00
Time:                        11:52:40   Log-Likelihood:                -22624.
No. Observations:               20640   AIC:                         4.527e+04
Df Residuals:                   20631   BIC:                         4.534e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -36.9419      0.659    -56.067      0.0

---
## sklearn vs statsmodels

Both libraries fit a linear regression, but they answer different questions.

**sklearn LinearRegression:**
- Fast, memory-efficient, integrates seamlessly into sklearn pipelines
- Produces no statistical output — only predictions and coefficients
- Use it for: prediction tasks, production models, ML pipelines

**statsmodels OLS:**
- Produces the full statistical summary: p-values, confidence intervals, diagnostic tests
- Follows the statistical modeling tradition similar to R's `lm()`
- Use it for: understanding relationships, academic or scientific work, regulatory contexts where inference is required

> **sklearn asks "how well does this model predict?" statsmodels asks "what does this model tell us about the relationship between variables?" These are different questions and both are valid.**

See **[ml_concepts/03_the_ml_workflow.ipynb](../ml_concepts/03_the_ml_workflow.ipynb)** for the broader distinction between prediction and inference in machine learning.

---
## Try it yourself

In [5]:
np.random.seed(42)
_N_SYNTH = 80
_X_synth = np.random.uniform(-3, 3, _N_SYNTH)
_TRUE_SLOPE, _TRUE_INTERCEPT = 1.5, 0.5
_Y_synth = _TRUE_SLOPE * _X_synth + _TRUE_INTERCEPT + np.random.normal(0, 1.2, _N_SYNTH)

out_math = Output()

slope_sl = FloatSlider(value=0.5, min=-3.0, max=3.0, step=0.1,
    description="Slope:", style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"))
intercept_sl = FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1,
    description="Intercept:", style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"))

def render_math(change=None):
    sl  = slope_sl.value
    ic  = intercept_sl.value
    y_hat_synth = sl * _X_synth + ic
    mse = float(np.mean((_Y_synth - y_hat_synth) ** 2))
    x_line = np.array([-3.2, 3.2])
    traces = [
        go.Scatter(x=_X_synth, y=_Y_synth, mode="markers",
            marker=dict(color=PALETTE["muted"], size=7, opacity=0.55), name="Data"),
        go.Scatter(x=x_line, y=sl * x_line + ic, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name=f"ŷ = {sl:.2f}·x + {ic:.2f}"),
    ]
    layout = base_layout(
        title=f"ŷ = {sl:.2f}·x + {ic:.2f}  |  MSE = {mse:.3f}",
        xaxis_title="x", yaxis_title="y")
    layout.update(
        xaxis=dict(range=[-3.5, 3.5]),
        yaxis=dict(range=[-8, 8]),
        annotations=[dict(
            x=0, y=7, xref="x", yref="y",
            text=f"<b>MSE = {mse:.3f}</b><br>True: slope={_TRUE_SLOPE}, intercept={_TRUE_INTERCEPT}",
            showarrow=False,
            font=dict(size=13, color=PALETTE["secondary"]),
            bgcolor="white", bordercolor=PALETTE["secondary"], borderwidth=1,
        )],
    )
    with out_math:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

slope_sl.observe(render_math, names="value")
intercept_sl.observe(render_math, names="value")
display(VBox([slope_sl, intercept_sl, out_math]))
render_math()

In [6]:
_ch = fetch_california_housing(as_frame=True)
_Xh, _yh = _ch.data, _ch.target.values
_rng = np.random.RandomState(42)
_idx_w2 = _rng.choice(len(_yh), 400, replace=False)

out_fit = Output()

feature_dd = Dropdown(
    options=_ch.feature_names, value="MedInc",
    description="Feature:", style={"description_width": "80px"},
    layout=widgets.Layout(width="360px"))
resid_dd = Dropdown(
    options=["Hide residuals", "Show residuals"], value="Hide residuals",
    description="Residuals:", style={"description_width": "85px"},
    layout=widgets.Layout(width="280px"))

def render_fit(change=None):
    feat = feature_dd.value
    show_resid = resid_dd.value == "Show residuals"
    xv = _Xh[feat].values
    lr1 = LinearRegression().fit(xv.reshape(-1, 1), _yh)
    yh  = lr1.predict(xv.reshape(-1, 1))
    r2   = r2_score(_yh, yh)
    rmse = np.sqrt(mean_squared_error(_yh, yh))
    x_line = np.linspace(float(xv.min()), float(xv.max()), 200)
    y_line = lr1.predict(x_line.reshape(-1, 1))
    traces = [
        go.Scatter(x=xv[_idx_w2], y=_yh[_idx_w2], mode="markers",
            marker=dict(color=PALETTE["muted"], size=5, opacity=0.4), name="Districts (sample)"),
        go.Scatter(x=x_line, y=y_line, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name=f"Fit line  R²={r2:.3f}  RMSE={rmse:.4f}"),
    ]
    if show_resid:
        sample = _idx_w2[:60]
        for i in sample:
            traces.append(go.Scatter(
                x=[xv[i], xv[i]], y=[_yh[i], float(yh[i])],
                mode="lines", line=dict(color=PALETTE["secondary"], width=1, dash="dot"),
                showlegend=False))
    layout = base_layout(
        title=f"{feat} vs House Value  |  R²={r2:.3f}  RMSE={rmse:.4f}",
        xaxis_title=feat, yaxis_title="Median House Value ($100k)")
    with out_fit:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

feature_dd.observe(render_fit, names="value")
resid_dd.observe(render_fit, names="value")
display(VBox([HBox([feature_dd, resid_dd]), out_fit]))
render_fit()

In [7]:
from scipy import stats as _sp_stats
import statsmodels.api as _sm_w3

_h_w3 = fetch_california_housing(as_frame=True)
_Xw3 = _sm_w3.add_constant(_h_w3.data.values[:2000])
_yw3 = _h_w3.target.values[:2000]
_model_w3 = _sm_w3.OLS(_yw3, _Xw3).fit()
_inf_w3   = _model_w3.get_influence()
_fit_w3   = _model_w3.fittedvalues
_res_w3   = _model_w3.resid
_sres_w3  = _inf_w3.resid_studentized_internal
_order_w3 = np.arange(len(_res_w3))
_corr_w3  = _h_w3.data.iloc[:2000].corr()

out_assm = Output()

assm_dd = Dropdown(
    options=["Linearity", "Homoscedasticity", "Normality", "Independence", "Multicollinearity"],
    value="Linearity",
    description="Assumption:", style={"description_width": "105px"},
    layout=widgets.Layout(width="420px"))

def render_assm(change=None):
    assumption = assm_dd.value
    if assumption == "Linearity":
        traces = [go.Scatter(x=_fit_w3, y=_res_w3, mode="markers",
            marker=dict(color=PALETTE["primary"], size=4, opacity=0.4), name="Residual")]
        layout = base_layout("Residuals vs Fitted — checks linearity",
            "Fitted values", "Residuals")
        note = "Random scatter around zero: assumption likely met"
        note_color = PALETTE["accent"]
        fig = go.Figure(data=traces, layout=layout)
        fig.add_hline(y=0, line_color=PALETTE["muted"], line_width=1.5, line_dash="dash")
    elif assumption == "Homoscedasticity":
        traces = [go.Scatter(x=_fit_w3, y=np.sqrt(np.abs(_sres_w3)), mode="markers",
            marker=dict(color=PALETTE["primary"], size=4, opacity=0.4))]
        layout = base_layout("Scale-Location — checks homoscedasticity",
            "Fitted values", "sqrt|Standardized residuals|")
        note = "Flat horizontal band: assumption likely met. Funnel shape: violation"
        note_color = PALETTE["accent"]
        fig = go.Figure(data=traces, layout=layout)
    elif assumption == "Normality":
        osm, osr = _sp_stats.probplot(_res_w3)[0]
        qs, qi = np.polyfit(osm, osr, 1)
        qx = np.array([float(osm.min()), float(osm.max())])
        traces = [
            go.Scatter(x=osm, y=osr, mode="markers",
                marker=dict(color=PALETTE["primary"], size=3, opacity=0.4), name="Residuals"),
            go.Scatter(x=qx, y=qs*qx+qi, mode="lines",
                line=dict(color=PALETTE["muted"], width=2, dash="dash"), name="Reference"),
        ]
        layout = base_layout("Q-Q Plot — checks normality of residuals",
            "Theoretical quantiles", "Sample quantiles")
        note = "Points on diagonal: assumption likely met. Heavy tails: check further"
        note_color = PALETTE["accent"]
        fig = go.Figure(data=traces, layout=layout)
    elif assumption == "Independence":
        traces = [go.Scatter(x=_order_w3, y=_res_w3, mode="markers",
            marker=dict(color=PALETTE["primary"], size=3, opacity=0.35))]
        layout = base_layout("Residuals vs Order — checks independence",
            "Observation order", "Residuals")
        note = "No trend over time: assumption likely met. Drift or cycles: autocorrelation present"
        note_color = PALETTE["accent"]
        fig = go.Figure(data=traces, layout=layout)
        fig.add_hline(y=0, line_color=PALETTE["muted"], line_width=1.5, line_dash="dash")
    else:
        z = _corr_w3.values
        feat_names = list(_corr_w3.columns)
        traces = [go.Heatmap(z=z, x=feat_names, y=feat_names,
            colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
            text=np.round(z, 2), texttemplate="%{text}",
            colorbar=dict(title="r"))]
        layout = base_layout("Correlation Heatmap — checks multicollinearity", "", "")
        note = "Values near ±1 between features: multicollinearity present. Check VIF for confirmation"
        note_color = PALETTE["secondary"]
        fig = go.Figure(data=traces, layout=layout)
        fig.update_layout(height=420)

    fig.add_annotation(
        x=0.01, y=0.01, xref="paper", yref="paper",
        text=f"<b>{note}</b>",
        showarrow=False,
        font=dict(size=12, color=note_color),
        bgcolor="white", bordercolor=note_color, borderwidth=1,
    )
    with out_assm:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

assm_dd.observe(render_assm, names="value")
display(VBox([assm_dd, out_assm]))
render_assm()

In [8]:
from scipy import stats as _sp_gen
from IPython.display import HTML

# Fixed generative parameters
_TRUE_INTERCEPT = 10.0
_TRUE_SLOPE = 2.0

# Fixed draws — never resampled between renders, so dragging moves the SAME points
np.random.seed(42)
_x_gen = np.random.uniform(-10, 10, 400)
_u_gen = np.random.uniform(0, 1, 400)   # uniform "ranks" for inverse-CDF sampling

_BELL_ANCHORS = [-7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5]
_BELL_A = 1.3   # horizontal bell amplitude in x-units

def _gen_sigma(x, sigma0, var_type):
    xa = np.asarray(x, dtype=float)
    if var_type == "Constant variance":
        return np.full_like(xa, sigma0)
    else:
        return sigma0 * (0.3 + 1.7 * (xa + 10.0) / 20.0)

def _gen_z(u, shape_type):
    if shape_type == "Normal errors":
        return _sp_gen.norm.ppf(u)
    else:
        raw = _sp_gen.skewnorm.ppf(u, a=6)
        return (raw - raw.mean()) / raw.std()

def _bell_curve_trace(x0, mu0, s0, shape_type):
    y_vals = np.linspace(mu0 - 3.5 * s0, mu0 + 3.5 * s0, 200)
    t = (y_vals - mu0) / s0
    if shape_type == "Normal errors":
        shape = np.exp(-0.5 * t ** 2)
    else:
        d = _sp_gen.skewnorm.pdf(t, a=6)
        shape = d / d.max()
    return go.Scatter(x=x0 + _BELL_A * shape, y=y_vals, mode="lines",
        line=dict(color=PALETTE["primary"], width=2.5),
        showlegend=False, hoverinfo="skip")

out_gen = Output()
_gen_initialized = False

noise_sl_gen = FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1,
    description="Noise σ:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"))
var_tg_gen = widgets.ToggleButtons(
    options=["Constant variance", "Increasing variance"],
    value="Constant variance")
shape_tg_gen = widgets.ToggleButtons(
    options=["Normal errors", "Right-skewed errors"],
    value="Normal errors")

def render_gen(change=None):
    global _gen_initialized
    sigma0     = noise_sl_gen.value
    var_type   = var_tg_gen.value
    shape_type = shape_tg_gen.value

    sig_x = _gen_sigma(_x_gen, sigma0, var_type)
    z     = _gen_z(_u_gen, shape_type)
    y_gen = _TRUE_INTERCEPT + _TRUE_SLOPE * _x_gen + sig_x * z

    mu_y = _TRUE_INTERCEPT + _TRUE_SLOPE * _x_gen
    R2   = float(np.clip(np.var(mu_y) / np.var(y_gen), 0.0, 1.0))

    traces = [
        go.Scatter(x=_x_gen, y=y_gen, mode="markers",
            marker=dict(color=PALETTE["muted"], size=4, opacity=0.4),
            name="Observed y"),
        go.Scatter(x=[-11.0, 11.0],
            y=[_TRUE_INTERCEPT + _TRUE_SLOPE * v for v in [-11.0, 11.0]],
            mode="lines", line=dict(color="#E03131", width=3, dash="dash"),
            name="E[y|x] = 10 + 2x"),
    ]

    for x0 in _BELL_ANCHORS:
        mu0 = _TRUE_INTERCEPT + _TRUE_SLOPE * x0
        s0  = float(_gen_sigma(np.array([x0]), sigma0, var_type)[0])
        traces.append(go.Scatter(
            x=[x0, x0], y=[mu0 - 3 * s0, mu0 + 3 * s0], mode="lines",
            line=dict(color="#bbbbbb", width=1, dash="dash"),
            showlegend=False, hoverinfo="skip"))
        traces.append(go.Scatter(
            x=[x0, x0 + _BELL_A], y=[mu0, mu0], mode="lines",
            line=dict(color="#8b0000", width=2, dash="dash"),
            showlegend=False, hoverinfo="skip"))
        traces.append(_bell_curve_trace(x0, mu0, s0, shape_type))

    fig = go.Figure(data=traces)
    fig.update_layout(base_layout(
        title="Generative View: at each x, y is a distribution centered on the line",
        xaxis_title="x", yaxis_title="y"))
    fig.update_layout(height=460, margin=dict(t=60, b=40, l=50, r=20), showlegend=True)
    fig.update_xaxes(range=[-11, 11])
    fig.update_yaxes(range=[-20, 45])

    noise_desc = "low" if sigma0 < 1.5 else ("moderate" if sigma0 <= 3.0 else "high")
    line3 = (
        "Variance is constant (homoscedasticity holds): every bell is the same width. "
        "A Scale-Location plot would show a flat band."
        if var_type == "Constant variance" else
        "Variance grows with x (heteroscedasticity): the bells fan into a funnel. "
        "A Scale-Location plot would slope upward, and prediction intervals should "
        "widen toward the right.")
    line4 = (
        "Errors are symmetric and normal: a Q-Q plot of residuals would fall on the diagonal."
        if shape_type == "Normal errors" else
        "Errors are right-skewed: the bells lean and a Q-Q plot would bow into an S-curve.")

    interp = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:14px 18px;border-radius:6px;margin-top:10px;line-height:1.65;color:#333;">'
        f'<p style="margin:0 0 6px 0;">Each blue curve is the distribution of y at that x. '
        f'The red line is the conditional mean E[y|x] = 10 + 2x; the grey points are random '
        f'draws from those distributions.</p>'
        f'<p style="margin:0 0 6px 0;">σ ≈ {sigma0:.1f} → {noise_desc} scatter. '
        f'The true line explains about {R2:.0%} of the variance in y; the rest is irreducible noise.</p>'
        f'<p style="margin:0 0 6px 0;">{line3}</p>'
        f'<p style="margin:0;">{line4}</p>'
        f'</div>'
    )

    with out_gen:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(interp))
    _gen_initialized = True

noise_sl_gen.observe(render_gen, names="value")
var_tg_gen.observe(render_gen, names="value")
shape_tg_gen.observe(render_gen, names="value")

display(VBox([noise_sl_gen, HBox([var_tg_gen, shape_tg_gen]), out_gen]))
render_gen()

---
## What's happening?

In the Math Explorer widget, every position of the slope and intercept sliders defines a line. The MSE value you see is the average of all the squared distances from each data point to that line. The moment you stop dragging, you have found a (slope, intercept) combination — but it is only optimal if you have found the global minimum of MSE, which forms a bowl-shaped surface in parameter space.

The Normal Equations jump directly to the bottom of that bowl. They work because MSE is a convex function of the parameters: it has exactly one minimum, no local traps, and the derivative is zero at the bottom. Setting the gradient to zero and solving gives the closed-form formula above.

In the boundary widget, the residuals (vertical lines) show you the errors the model is still making. A good fit has small residuals scattered randomly above and below the line. If you see a curved pattern in the residuals — errors that are systematically positive on one side and negative on the other — that is a sign that the linear assumption is wrong and you need polynomial features or a different model family.

---
## Key hyperparameters

**`fit_intercept`** (default `True`) — whether to add an intercept term $\theta_0$. Set to `False` only when you know the line must pass through the origin (rare in practice).

**`positive`** (default `False`) — constrains all coefficients to be non-negative. Useful in dose-response models or when negative effects are physically impossible.

**`copy_X`** (default `True`) — whether to copy the input matrix before fitting. Keeping this `True` prevents sklearn from modifying your original array.

For the full list of hyperparameters, see the sklearn documentation:
[https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

---
## Strengths and weaknesses

| Strengths | Weaknesses |
|-----------|------------|
| Extremely interpretable — every coefficient has a direct meaning | Assumes a linear relationship between features and target |
| Fast to train even on very large datasets | Sensitive to outliers (squared errors amplify large residuals) |
| Closed-form solution: no learning rate, no iterations | Underfits when the true relationship is nonlinear |
| Coefficients double as feature importance scores | Assumes no multicollinearity between features |
| Strong baseline that is hard to beat on truly linear problems | Prediction intervals require homoscedasticity assumption |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| You expect a roughly linear relationship between features and target | The scatter plot shows a clear curve or nonlinear pattern |
| Interpretability is important (regulated industries, A/B test analysis) | Your features are highly correlated (multicollinearity inflates coefficients) |
| You need a fast, debuggable baseline to start from | You have many irrelevant features (use Ridge or Lasso instead) |
| Your dataset is small and a complex model would overfit | The residuals show a systematic pattern (try polynomial or a different family) |
| You want to understand the direction and size of each feature's effect | You need probability outputs (use logistic regression instead) |

In [9]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print()
print("Coefficients (feature → weight):")
for feat, coef in zip(housing.feature_names, model.coef_):
    print(f"  {feat:25s}  {coef:+.4f}")
print(f"  {'intercept':25s}  {model.intercept_:+.4f}")

R²:   0.5758
RMSE: 0.7456

Coefficients (feature → weight):
  MedInc                     +0.4487
  HouseAge                   +0.0097
  AveRooms                   -0.1233
  AveBedrms                  +0.7831
  Population                 -0.0000
  AveOccup                   -0.0035
  Latitude                   -0.4198
  Longitude                  -0.4337
  intercept                  -37.0233


---
## Real-world example: Median income predicts house value

The California Housing dataset contains census information for 20,640 districts in California. The single strongest predictor of median house value is **median income** — districts with higher earners have more expensive homes, and the relationship is roughly linear over a wide range.

- **Notice:** The fit line captures the broad upward trend, but many points scatter widely above and below — median income alone explains only part of the story
- **Notice:** The relationship is steepest at lower income levels; very high incomes have a flatter effect, suggesting the linear model slightly misrepresents the top end
- **Notice:** A few very high-value districts ($5 cap) are all clustered at the top regardless of income — those are data ceiling artifacts, not real patterns

> **Discussion question:** The model sees a positive slope. Does that mean income *causes* house prices to be high, or is something else going on?

### Where linear regression is used in practice

| Industry | Application | Target variable |
|---|---|---|
| Real estate | Automated valuation models | Property price |
| Finance | Credit scoring (regulatory baseline) | Probability of default |
| Marketing | Budget attribution | Revenue attributed to channel |
| Healthcare | Epidemiology | Predicted disease rate per 1,000 |
| Energy | Demand forecasting | kWh consumption |

In [10]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
x_all = housing.data["MedInc"].values
y_all = housing.target.values

model_1d = LinearRegression()
model_1d.fit(x_all.reshape(-1, 1), y_all)
x_line = np.linspace(x_all.min(), x_all.max(), 300)
y_line = model_1d.predict(x_line.reshape(-1, 1))

idx = np.random.choice(len(x_all), 500, replace=False)

fig = go.Figure(data=[
    go.Scatter(
        x=x_all[idx], y=y_all[idx], mode="markers",
        marker=dict(color=PALETTE["muted"], size=5, opacity=0.45),
        name="District sample (n=500)",
    ),
    go.Scatter(
        x=x_line, y=y_line, mode="lines",
        line=dict(color=PALETTE["secondary"], width=3),
        name=(f"ŷ = {model_1d.coef_[0]:.3f}·MedInc "
              f"+ {model_1d.intercept_:.3f}"),
    ),
], layout=base_layout(
    title="Median Income vs Median House Value — California Housing",
    xaxis_title="Median Income (tens of thousands $)",
    yaxis_title="Median House Value (hundreds of thousands $)",
))
fig.show()

> **Linear regression finds the one straight line that minimizes the sum of squared residuals — the coefficients tell you exactly how much each feature moves the prediction, making it the most interpretable model in machine learning.**

---
*Next up: 02 — Multiple Linear Regression, extending the line to many features at once*